# Advanced Problems with Solutions: Generators and Context Managers

This notebook turns the generator/context-manager topic into a progressive problem set with complete, runnable solutions.

## Topics covered

- manual generator lifecycle: `next`, `throw`, and `close`;
- the generator-based context-manager protocol;
- the exactly-one-`yield` rule;
- robust cleanup with `try` / `finally`;
- exception propagation, suppression, translation, and annotation;
- `contextlib.contextmanager` and `ContextDecorator` behavior;
- nested managers and cleanup order;
- dynamic resource management with `ExitStack`;
- rollback and transactional patterns;
- atomic file replacement;
- asynchronous context managers and `AsyncExitStack`;
- debugging common generator-context-manager mistakes.

Every problem contains a solution and assertions. The notebook uses temporary files so it is safe to run repeatedly.

## Best-practice checklist

1. Prefer `contextlib.contextmanager` to a hand-written generator wrapper in application code.
2. Acquire before `yield`; clean up in `finally`.
3. Yield exactly once.
4. Suppress only exceptions you explicitly understand.
5. Re-raise with bare `raise` to preserve the traceback.
6. Use `raise NewError(...) from exc` when translating exceptions.
7. Treat each generator context-manager instance as single-use.
8. Use `ExitStack` when the number of resources is determined at runtime.
9. Test normal exit, exceptional exit, acquisition failure, and partial acquisition.
10. Prefer deterministic assertions over timing-sensitive expectations.

## Setup

In [1]:
from __future__ import annotations

import asyncio
import contextlib
import decimal
import io
import os
import sys
import tempfile
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, AsyncIterator, Callable, Iterator

TEST_ROOT = Path(tempfile.mkdtemp(prefix='advanced_generator_cm_'))
print('Workspace:', TEST_ROOT)

Workspace: C:\Users\user1\AppData\Local\Temp\advanced_generator_cm_it4wo8m4


## Problem 1 — Drive a generator manually

Create a generator that acquires a dictionary-like resource, yields it, and releases it in `finally`. Drive it with `next()` and verify the two phases.

### Solution

In [2]:
def traced_resource(name: str) -> Iterator[dict[str, Any]]:
    print(f'acquire:{name}')
    resource = {'name': name, 'open': True}
    try:
        yield resource
    finally:
        resource['open'] = False
        print(f'release:{name}')


gen = traced_resource('cache')
resource = next(gen)
assert resource == {'name': 'cache', 'open': True}

try:
    next(gen)
except StopIteration:
    pass
else:
    raise AssertionError('The generator should terminate after cleanup.')

assert resource['open'] is False

acquire:cache
release:cache


## Problem 2 — Prove that `close()` runs cleanup

A consumer can abandon a suspended generator. Show that `close()` still executes its `finally` block.

### Solution

In [3]:
events: list[str] = []

def closable() -> Iterator[str]:
    events.append('acquired')
    try:
        yield 'resource'
    finally:
        events.append('released')


g = closable()
assert next(g) == 'resource'
g.close()
assert events == ['acquired', 'released']
assert g.gi_frame is None

## Problem 3 — Inspect exception injection with `throw()`

Throw a `ValueError` into a suspended generator. The generator should catch it, record it, and then terminate.

### Solution

In [ ]:
throw_log: list[str] = []

def receiver() -> Iterator[str]:
    try:
        yield 'ready'
    except ValueError as exc:
        throw_log.append(f'caught:{exc}')
    finally:
        throw_log.append('cleanup')


g = receiver()
assert next(g) == 'ready'
try:
    g.throw(ValueError, ValueError('bad value'))
except StopIteration:
    pass

assert throw_log == ['caught:bad value', 'cleanup']

## Problem 4 — Implement a minimal generator context manager

Support a generator function plus positional and keyword arguments. Handle only the normal exit path for now.

### Solution

In [5]:
class MinimalGenContextManager:
    def __init__(self, generator_function: Callable[..., Iterator[Any]], *args: Any, **kwargs: Any):
        self._generator = generator_function(*args, **kwargs)

    def __enter__(self) -> Any:
        return next(self._generator)

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        try:
            next(self._generator)
        except StopIteration:
            return False
        raise RuntimeError('generator did not stop')


def generated_list(start: int, stop: int) -> Iterator[list[int]]:
    values = list(range(start, stop))
    try:
        yield values
    finally:
        values.clear()


with MinimalGenContextManager(generated_list, 2, 6) as values:
    assert values == [2, 3, 4, 5]

assert values == []

## Problem 5 — Detect a generator that never yields

Convert the raw `StopIteration` raised during `__enter__` into a clear protocol error.

### Solution

In [6]:
def never_yields() -> Iterator[None]:
    if False:
        yield None


class CheckedEnterContextManager(MinimalGenContextManager):
    def __enter__(self) -> Any:
        try:
            return next(self._generator)
        except StopIteration as exc:
            raise RuntimeError('generator did not yield') from exc


try:
    with CheckedEnterContextManager(never_yields):
        pass
except RuntimeError as exc:
    assert str(exc) == 'generator did not yield'
else:
    raise AssertionError('A generator that never yields must be rejected.')

## Problem 6 — Enforce exactly one `yield`

Use a broken generator that yields twice. Verify that leaving the block reports a protocol error.

### Solution

In [7]:
def yields_twice() -> Iterator[str]:
    yield 'first'
    yield 'second'


try:
    with MinimalGenContextManager(yields_twice) as value:
        assert value == 'first'
except RuntimeError as exc:
    assert str(exc) == 'generator did not stop'
else:
    raise AssertionError('A second yield must be rejected.')

## Problem 7 — Implement exception-aware `__exit__`

Throw body exceptions into the generator. Suppress an exception only when the generator catches it and terminates normally.

### Solution

In [8]:
class RobustGeneratorContextManager:
    def __init__(self, generator_function: Callable[..., Iterator[Any]], *args: Any, **kwargs: Any):
        self._generator = generator_function(*args, **kwargs)

    def __enter__(self) -> Any:
        try:
            return next(self._generator)
        except StopIteration as exc:
            raise RuntimeError('generator did not yield') from exc

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if exc_type is None:
            try:
                next(self._generator)
            except StopIteration:
                return False
            raise RuntimeError('generator did not stop')

        if exc_value is None:
            exc_value = exc_type()

        try:
            self._generator.throw(exc_type, exc_value, traceback)
        except StopIteration:
            return True
        except exc_type as raised:
            if raised is exc_value:
                return False
            raise
        raise RuntimeError('generator did not stop after throw')


def suppress_lookup_error() -> Iterator[str]:
    try:
        yield 'ready'
    except LookupError:
        print('LookupError handled by generator')


with RobustGeneratorContextManager(suppress_lookup_error) as state:
    assert state == 'ready'
    raise KeyError('missing')

LookupError handled by generator


C:\Users\user1\AppData\Local\Temp\ipykernel_14120\269552792.py:23: DeprecationWarning: the (type, exc, tb) signature of throw() is deprecated, use the single-arg signature instead.
  self._generator.throw(exc_type, exc_value, traceback)


## Problem 8 — Propagate an unhandled exception unchanged

A cleanup-only generator must not suppress an exception from the body.

### Solution

In [9]:
def cleanup_only(log: list[str]) -> Iterator[None]:
    try:
        yield
    finally:
        log.append('cleaned')


cleanup_log: list[str] = []
original = ValueError('invalid payload')

try:
    with RobustGeneratorContextManager(cleanup_only, cleanup_log):
        raise original
except ValueError as exc:
    assert exc is original
else:
    raise AssertionError('The original exception should propagate.')

assert cleanup_log == ['cleaned']

C:\Users\user1\AppData\Local\Temp\ipykernel_14120\269552792.py:23: DeprecationWarning: the (type, exc, tb) signature of throw() is deprecated, use the single-arg signature instead.
  self._generator.throw(exc_type, exc_value, traceback)


## Problem 9 — Rewrite with `@contextmanager`

Create a safe text-file manager. It must close the handle on normal and exceptional exits.

### Solution

In [10]:
@contextlib.contextmanager
def open_text(path: Path, mode: str = 'r', encoding: str = 'utf-8') -> Iterator[io.TextIOBase]:
    handle = open(path, mode, encoding=encoding)
    try:
        yield handle
    finally:
        handle.close()


sample_path = TEST_ROOT / 'sample.txt'
with open_text(sample_path, 'w') as handle:
    handle.write('generator context manager')

assert handle.closed
assert sample_path.read_text(encoding='utf-8') == 'generator context manager'

## Problem 10 — Guarantee closure after a body exception

Raise inside `open_text` and verify both cleanup and exception propagation.

### Solution

In [11]:
exception_path = TEST_ROOT / 'exception.txt'
try:
    with open_text(exception_path, 'w') as handle:
        handle.write('partial')
        raise RuntimeError('body failed')
except RuntimeError as exc:
    assert str(exc) == 'body failed'
else:
    raise AssertionError('The body exception should propagate.')

assert handle.closed
assert exception_path.read_text(encoding='utf-8') == 'partial'

## Problem 11 — Restore mutable state exactly

Temporarily append values to a list, then restore the original list even if the body changes existing elements.

### Solution

In [12]:
@contextlib.contextmanager
def temporary_append(target: list[Any], *items: Any) -> Iterator[list[Any]]:
    snapshot = target.copy()
    target.extend(items)
    try:
        yield target
    finally:
        target[:] = snapshot


numbers = [1, 2]
try:
    with temporary_append(numbers, 3, 4) as active:
        active[0] = 999
        active.append(5)
        assert active == [999, 2, 3, 4, 5]
        raise RuntimeError('force rollback')
except RuntimeError:
    pass

assert numbers == [1, 2]

## Problem 12 — Suppress only allowed exception types

Implement a selective suppression manager and prove that unrelated exceptions still escape.

### Solution

In [13]:
@contextlib.contextmanager
def suppress_only(*allowed: type[BaseException]) -> Iterator[None]:
    try:
        yield
    except allowed:
        pass


with suppress_only(FileNotFoundError):
    raise FileNotFoundError('optional input')

try:
    with suppress_only(FileNotFoundError):
        raise PermissionError('permission problem')
except PermissionError:
    pass
else:
    raise AssertionError('PermissionError must not be suppressed.')

## Problem 13 — Translate exceptions with chaining

Convert `KeyError` into a domain-specific error while preserving the original cause.

### Solution

In [14]:
class ConfigurationError(RuntimeError):
    pass


@contextlib.contextmanager
def translate_missing_key() -> Iterator[None]:
    try:
        yield
    except KeyError as exc:
        raise ConfigurationError(f'Missing configuration key: {exc.args[0]}') from exc


try:
    with translate_missing_key():
        settings = {'host': 'localhost'}
        settings['port']
except ConfigurationError as exc:
    assert 'port' in str(exc)
    assert isinstance(exc.__cause__, KeyError)
else:
    raise AssertionError('KeyError should be translated.')

## Problem 14 — Return a result finalized on exit

Yield a timing result object whose `elapsed` field is populated after the block finishes.

### Solution

In [15]:
@dataclass
class TimingResult:
    started_at: float = 0.0
    ended_at: float = 0.0
    elapsed: float = 0.0


@contextlib.contextmanager
def timer() -> Iterator[TimingResult]:
    result = TimingResult(started_at=time.perf_counter())
    try:
        yield result
    finally:
        result.ended_at = time.perf_counter()
        result.elapsed = result.ended_at - result.started_at


with timer() as timing:
    sum(range(100_000))

assert timing.elapsed >= 0
assert timing.ended_at >= timing.started_at

## Problem 15 — Redirect `stdout` safely

Temporarily replace `sys.stdout`, return the destination stream, and always restore the original stream.

### Solution

In [16]:
@contextlib.contextmanager
def redirect_stdout_to(stream: io.TextIOBase) -> Iterator[io.TextIOBase]:
    previous = sys.stdout
    sys.stdout = stream
    try:
        yield stream
    finally:
        sys.stdout = previous


buffer = io.StringIO()
with redirect_stdout_to(buffer):
    print('captured line 1')
    print('captured line 2')

assert buffer.getvalue() == 'captured line 1\ncaptured line 2\n'
print('stdout restored')

stdout restored


## Problem 16 — Compose context managers

Wrap `open_text` with logging. Verify the nested file manager closes the handle before the outer manager returns.

### Solution

In [17]:
@contextlib.contextmanager
def logged_open_text(path: Path, mode: str, log: list[str]) -> Iterator[io.TextIOBase]:
    log.append(f'opening:{path.name}')
    with open_text(path, mode) as handle:
        try:
            yield handle
        finally:
            log.append(f'leaving-body:{path.name}')
    log.append(f'closed:{path.name}')


events = []
logged_path = TEST_ROOT / 'logged.txt'
with logged_open_text(logged_path, 'w', events) as handle:
    handle.write('data')
    assert not handle.closed

assert handle.closed
assert events == ['opening:logged.txt', 'leaving-body:logged.txt', 'closed:logged.txt']

## Problem 17 — Single-use instance, reusable factory

Show that repeated calls to the decorated function are valid, while re-entering the same returned manager instance is invalid.

### Solution

In [18]:
@contextlib.contextmanager
def marker(log: list[str]) -> Iterator[None]:
    log.append('enter')
    try:
        yield
    finally:
        log.append('exit')


reuse_log: list[str] = []
with marker(reuse_log):
    pass
with marker(reuse_log):
    pass
assert reuse_log == ['enter', 'exit', 'enter', 'exit']

single_instance = marker([])
with single_instance:
    pass

try:
    with single_instance:
        pass
except (AttributeError, RuntimeError):
    pass
else:
    raise AssertionError('A generator context-manager instance is single-use.')

## Problem 18 — Use a generator context manager as a decorator

Decorate a function and record entry, body, and exit for each call.

### Solution

In [19]:
decorator_log: list[str] = []

@contextlib.contextmanager
def audit_call(label: str) -> Iterator[None]:
    decorator_log.append(f'start:{label}')
    try:
        yield
    finally:
        decorator_log.append(f'stop:{label}')


@audit_call('calculate')
def calculate_total(values: list[int]) -> int:
    decorator_log.append('body')
    return sum(values)


assert calculate_total([10, 20, 30]) == 60
assert calculate_total([1, 2]) == 3
assert decorator_log == [
    'start:calculate', 'body', 'stop:calculate',
    'start:calculate', 'body', 'stop:calculate',
]

## Problem 19 — Verify last-in, first-out cleanup

Nest three managers and prove their exit order.

### Solution

In [20]:
@contextlib.contextmanager
def event_scope(name: str, log: list[str]) -> Iterator[str]:
    log.append(f'enter:{name}')
    try:
        yield name
    finally:
        log.append(f'exit:{name}')


nesting_log: list[str] = []
with event_scope('A', nesting_log):
    with event_scope('B', nesting_log):
        with event_scope('C', nesting_log):
            nesting_log.append('body')

assert nesting_log == [
    'enter:A', 'enter:B', 'enter:C', 'body',
    'exit:C', 'exit:B', 'exit:A',
]

## Problem 20 — Open a runtime-sized file collection with `ExitStack`

Open an arbitrary number of files and ensure every handle closes automatically.

### Solution

In [21]:
paths = [TEST_ROOT / f'part_{index}.txt' for index in range(5)]

with contextlib.ExitStack() as stack:
    handles = [stack.enter_context(open_text(path, 'w')) for path in paths]
    for index, handle in enumerate(handles):
        handle.write(f'payload-{index}')

assert all(handle.closed for handle in handles)
assert [path.read_text(encoding='utf-8') for path in paths] == [
    'payload-0', 'payload-1', 'payload-2', 'payload-3', 'payload-4'
]

## Problem 21 — Register rollback callbacks

Use `ExitStack.callback` to build a rollback plan. On failure, callbacks must run in reverse order. On commit, cancel them with `pop_all()`.

### Solution

In [22]:
rollback_log: list[str] = []

def undo_step(name: str) -> None:
    rollback_log.append(f'undo:{name}')


try:
    with contextlib.ExitStack() as stack:
        stack.callback(undo_step, 'directory')
        stack.callback(undo_step, 'file')
        stack.callback(undo_step, 'metadata')
        raise RuntimeError('operation failed')
except RuntimeError:
    pass

assert rollback_log == ['undo:metadata', 'undo:file', 'undo:directory']

commit_log: list[str] = []
with contextlib.ExitStack() as stack:
    stack.callback(commit_log.append, 'rolled back')
    stack.pop_all()

assert commit_log == []

## Problem 22 — Build an atomic text writer

Write to a sibling temporary file. Replace the destination only after successful completion; delete temporary output on failure.

### Solution

In [23]:
@contextlib.contextmanager
def atomic_text_writer(destination: Path, encoding: str = 'utf-8') -> Iterator[io.TextIOBase]:
    destination = Path(destination)
    temporary = destination.with_name(destination.name + '.tmp')
    handle = open(temporary, 'w', encoding=encoding)

    try:
        yield handle
        handle.flush()
        os.fsync(handle.fileno())
        handle.close()
        os.replace(temporary, destination)
    except BaseException:
        if not handle.closed:
            handle.close()
        temporary.unlink(missing_ok=True)
        raise


atomic_path = TEST_ROOT / 'atomic.txt'
atomic_path.write_text('old', encoding='utf-8')

try:
    with atomic_text_writer(atomic_path) as handle:
        handle.write('incomplete')
        raise ValueError('abort')
except ValueError:
    pass

assert atomic_path.read_text(encoding='utf-8') == 'old'
assert not atomic_path.with_name('atomic.txt.tmp').exists()

with atomic_text_writer(atomic_path) as handle:
    handle.write('new')

assert atomic_path.read_text(encoding='utf-8') == 'new'

## Problem 23 — Implement a dictionary transaction

Work on a copy. Commit only on normal exit; otherwise leave the original dictionary untouched.

### Solution

In [24]:
@contextlib.contextmanager
def dictionary_transaction(target: dict[str, Any]) -> Iterator[dict[str, Any]]:
    working = target.copy()
    try:
        yield working
    except Exception:
        raise
    else:
        target.clear()
        target.update(working)


account = {'balance': 100, 'status': 'active'}
try:
    with dictionary_transaction(account) as working:
        working['balance'] -= 30
        working['status'] = 'pending'
        raise RuntimeError('payment failure')
except RuntimeError:
    pass

assert account == {'balance': 100, 'status': 'active'}

with dictionary_transaction(account) as working:
    working['balance'] -= 30
    working['status'] = 'paid'

assert account == {'balance': 70, 'status': 'paid'}

## Problem 24 — Temporarily change decimal policy

Use `decimal.localcontext()` inside a generator manager so precision and rounding are restored automatically.

### Solution

In [25]:
@contextlib.contextmanager
def decimal_policy(precision: int, rounding: str = decimal.ROUND_HALF_EVEN) -> Iterator[decimal.Context]:
    with decimal.localcontext() as context:
        context.prec = precision
        context.rounding = rounding
        yield context


outer_signature = (decimal.getcontext().prec, decimal.getcontext().rounding)
with decimal_policy(5, decimal.ROUND_DOWN):
    result = decimal.Decimal(1) / decimal.Decimal(7)
    assert str(result) == '0.14285'
    assert decimal.getcontext().rounding == decimal.ROUND_DOWN

assert (decimal.getcontext().prec, decimal.getcontext().rounding) == outer_signature

## Problem 25 — Restore environment variables exactly

Use `None` to mean “temporarily remove this variable.” Restore both missing and existing variables to their original states.

### Solution

In [26]:
_MISSING = object()

@contextlib.contextmanager
def temporary_environment(**changes: str | None) -> Iterator[None]:
    previous: dict[str, object | str] = {
        key: os.environ.get(key, _MISSING)
        for key in changes
    }

    for key, value in changes.items():
        if value is None:
            os.environ.pop(key, None)
        else:
            os.environ[key] = value

    try:
        yield
    finally:
        for key, old_value in previous.items():
            if old_value is _MISSING:
                os.environ.pop(key, None)
            else:
                os.environ[key] = str(old_value)


os.environ['COURSE_EXISTING'] = 'original'
os.environ.pop('COURSE_NEW', None)

with temporary_environment(COURSE_EXISTING='temporary', COURSE_NEW='created'):
    assert os.environ['COURSE_EXISTING'] == 'temporary'
    assert os.environ['COURSE_NEW'] == 'created'

assert os.environ['COURSE_EXISTING'] == 'original'
assert 'COURSE_NEW' not in os.environ
os.environ.pop('COURSE_EXISTING', None)

'original'

## Problem 26 — Avoid cleanup after failed acquisition

Place acquisition before the `try` that owns cleanup. This prevents cleanup code from referencing an unassigned resource.

### Solution

In [27]:
class DemoResource:
    def __init__(self, name: str):
        self.name = name
        self.closed = False

    def close(self) -> None:
        self.closed = True


def acquire_resource(name: str, *, fail: bool = False) -> DemoResource:
    if fail:
        raise OSError('acquisition failed')
    return DemoResource(name)


@contextlib.contextmanager
def managed_demo_resource(name: str, *, fail: bool = False) -> Iterator[DemoResource]:
    resource = acquire_resource(name, fail=fail)
    try:
        yield resource
    finally:
        resource.close()


try:
    with managed_demo_resource('broken', fail=True):
        pass
except OSError as exc:
    assert str(exc) == 'acquisition failed'
else:
    raise AssertionError('Acquisition failure should propagate.')

with managed_demo_resource('working') as demo:
    assert not demo.closed
assert demo.closed

## Problem 27 — Annotate and re-raise the same exception

Add diagnostic context without changing exception identity or traceback.

### Solution

In [28]:
@contextlib.contextmanager
def annotate_errors(note: str) -> Iterator[None]:
    try:
        yield
    except Exception as exc:
        add_note = getattr(exc, 'add_note', None)
        if add_note is not None:
            add_note(note)
        raise


problem = RuntimeError('service unavailable')
try:
    with annotate_errors('while refreshing the cache'):
        raise problem
except RuntimeError as exc:
    assert exc is problem
    notes = getattr(exc, '__notes__', [])
    if notes:
        assert 'while refreshing the cache' in notes
else:
    raise AssertionError('The exception should propagate.')

## Problem 28 — Debug an illegal second yield after an exception

A generator context manager must not yield a fallback value after the body starts. Return a mutable state before the body and update it when an allowed exception occurs.

### Solution

In [29]:
@dataclass
class OperationState:
    value: str
    failed: bool = False


@contextlib.contextmanager
def operation_state(default: str) -> Iterator[OperationState]:
    state = OperationState(default)
    try:
        yield state
    except LookupError:
        state.failed = True
        # No second yield: returning ends the generator and suppresses LookupError.


with operation_state('fallback') as state:
    raise KeyError('missing optional value')

assert state.value == 'fallback'
assert state.failed is True

## Problem 29 — Write an asynchronous generator context manager

Connect before yielding and disconnect in `finally`, including exceptional exits.

### Solution

In [30]:
@dataclass
class AsyncConnection:
    name: str
    connected: bool = False
    messages: list[str] = field(default_factory=list)

    async def connect(self) -> None:
        await asyncio.sleep(0)
        self.connected = True

    async def send(self, message: str) -> None:
        if not self.connected:
            raise RuntimeError('not connected')
        await asyncio.sleep(0)
        self.messages.append(message)

    async def disconnect(self) -> None:
        await asyncio.sleep(0)
        self.connected = False


@contextlib.asynccontextmanager
async def open_async_connection(name: str) -> AsyncIterator[AsyncConnection]:
    connection = AsyncConnection(name)
    await connection.connect()
    try:
        yield connection
    finally:
        await connection.disconnect()


async def async_context_demo() -> None:
    try:
        async with open_async_connection('events') as connection:
            assert connection.connected
            await connection.send('hello')
            raise ValueError('body failure')
    except ValueError:
        pass

    assert connection.messages == ['hello']
    assert not connection.connected


await async_context_demo()

## Problem 30 — Manage dynamic async resources with `AsyncExitStack`

Acquire several asynchronous managers at runtime and verify reverse-order cleanup.

### Solution

In [31]:
async_cleanup_log: list[str] = []

@contextlib.asynccontextmanager
async def logged_async_resource(name: str) -> AsyncIterator[str]:
    async_cleanup_log.append(f'enter:{name}')
    try:
        yield name
    finally:
        await asyncio.sleep(0)
        async_cleanup_log.append(f'exit:{name}')


async def async_stack_demo() -> None:
    async with contextlib.AsyncExitStack() as stack:
        resources = [
            await stack.enter_async_context(logged_async_resource(name))
            for name in ['A', 'B', 'C']
        ]
        assert resources == ['A', 'B', 'C']
        async_cleanup_log.append('body')


await async_stack_demo()
assert async_cleanup_log == [
    'enter:A', 'enter:B', 'enter:C', 'body',
    'exit:C', 'exit:B', 'exit:A',
]

## Problem 31 — Capstone: safe multi-file report session

Create a report session that combines:

- a staging directory;
- a helper object for writing report files;
- publication only after successful body completion;
- automatic staging cleanup;
- a timing result returned alongside the writer.

This is a staged publication pattern, not a full filesystem transaction across multiple independent replacements.

### Solution

In [32]:
@dataclass
class ReportWriter:
    staging_directory: Path

    def write(self, relative_name: str, content: str) -> Path:
        path = self.staging_directory / relative_name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding='utf-8')
        return path


@dataclass
class ReportSession:
    writer: ReportWriter
    timing: TimingResult


@contextlib.contextmanager
def report_session(destination: Path) -> Iterator[ReportSession]:
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory(prefix='.staging-', dir=destination.parent) as temp_dir:
        with timer() as timing:
            writer = ReportWriter(Path(temp_dir))
            yield ReportSession(writer=writer, timing=timing)

        staged_files = [path for path in writer.staging_directory.rglob('*') if path.is_file()]
        for source in staged_files:
            relative = source.relative_to(writer.staging_directory)
            target = destination / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            with atomic_text_writer(target) as handle:
                handle.write(source.read_text(encoding='utf-8'))


published = TEST_ROOT / 'published'
published.mkdir(exist_ok=True)
(published / 'summary.txt').write_text('old summary', encoding='utf-8')

try:
    with report_session(published) as session:
        session.writer.write('summary.txt', 'aborted summary')
        session.writer.write('details/data.txt', 'temporary')
        raise RuntimeError('report failed')
except RuntimeError:
    pass

assert (published / 'summary.txt').read_text(encoding='utf-8') == 'old summary'
assert not (published / 'details/data.txt').exists()

with report_session(published) as session:
    session.writer.write('summary.txt', 'final summary')
    session.writer.write('details/data.txt', '42')

assert session.timing.elapsed >= 0
assert (published / 'summary.txt').read_text(encoding='utf-8') == 'final summary'
assert (published / 'details/data.txt').read_text(encoding='utf-8') == '42'

# Final review

## Generator-based context-manager protocol

- Code before `yield` corresponds to acquisition / `__enter__`.
- The yielded object is bound by `as`.
- Code after `yield` corresponds to exit handling.
- A body exception is injected at the suspended `yield`.
- `finally` is the correct location for unconditional cleanup.
- Normal generator termination completes the context.
- A missing yield or second yield is a protocol error.

## Tool selection

- Use `@contextmanager` for compact synchronous managers.
- Use `@asynccontextmanager` for asynchronous acquisition and cleanup.
- Use `ExitStack` or `AsyncExitStack` for dynamic resources.
- Prefer a class when you need sophisticated reusable or re-entrant state.

In [33]:
required_artifacts = [
    sample_path,
    logged_path,
    atomic_path,
    published / 'summary.txt',
    published / 'details/data.txt',
]
assert all(path.exists() for path in required_artifacts)
print('All 31 solved problems executed successfully.')

All 31 solved problems executed successfully.
